<a href="https://colab.research.google.com/github/kingc007/DataBootcamp/blob/main/KaggleProject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import make_column_transformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

In [7]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

In [8]:
X = train.drop(columns=["Id", "SalePrice"])
y = train["SalePrice"]
X_test = test.drop(columns=["Id"])

In [9]:
#Group the columns by type
num_cols = ["OverallQual", "GrLivArea", "TotalBsmtSF", "GarageArea", "YearBuilt", "FullBath", "LotFrontage"]
cat_cols = ["CentralAir", "KitchenQual"]

In [10]:
X = train[num_cols + cat_cols]
y = train["SalePrice"]
X_test = test[num_cols + cat_cols]

In [11]:
# Prepare the columns so the model can use them.
# SimpleImputer fills blanks in the number columns (LotFrontage has some) with the column average.
# OneHotEncoder turns text categories (like "Y"/"N") into 0/1 columns.
transformer = make_column_transformer(
    (SimpleImputer(), num_cols),
    (OneHotEncoder(), cat_cols),
)

In [12]:
# Build the model. A Pipeline runs the steps in order:
# first prepare the columns, then fit a linear regression.
model_pipeline = Pipeline([
    ("transform", transformer),
    ("model", LinearRegression()),
])

In [18]:
# Test the model on houses it has not seen.
# Split train into 80% to learn from and 20% to check against.
# RMSE is the typical size of our mistake in dollars (smaller is better).
# Its value shows under the cell when it runs.X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

model_pipeline.fit(X_train, y_train)
val_pred = model_pipeline.predict(X_val)

rmse = np.sqrt(mean_squared_error(y_val, val_pred))
rmse

#looking at leaderboard, not sure how we got to that 20k away, this is our actual model too

np.float64(41448.73504971537)

In [19]:
# Learn from ALL of train, predict the test houses, and save the answers to preds.csv.
model_pipeline.fit(X, y)
preds = model_pipeline.predict(X_test)

pdf = pd.DataFrame(preds, columns=["SalePrice"])
pdf["Id"] = test["Id"]
pdf.to_csv("preds.csv", index=False)